In [2]:
# import necessary packages
import requests
import json
from pathlib import Path
import time

In [3]:
# define draft year ranges
# picked these draft years because 1979 is the first year of data available and 2018
# gives the younger players a chance to have developed and put up respectable numbers
draft_year_start = 1979
draft_year_end = 2018

# seasons to look at for the draft classes
# used in getting the bios
season_year_start = 19791980
season_year_end = 20252026

# seasons to look at for the stats
season_stats_year_start = 1979
season_stats_year_end = 2025



# define the file path to save the data
draft_picks_path = Path("../data/raw/draft_picks_1979_to_2018.json")
skater_bios_path = Path("../data/raw/skater_bios_1979-1980_to_2025-2026.json")
skater_stats_path = Path("../data/raw/skater_stats_1979-1980_to_2025-2026.json")


In [4]:
season_year_start = 19791980
print(season_year_start // 10000)

1979


Grab the draft picks data

In [5]:
# draft_picks list to hold the data
draft_picks = []

# fetch the draft picks data from the NHL API if the file does not exist
# do it year by year to avoid overwhelming the API and to handle potential rate limits

# if the draft picks file does not exist, fetch the data from the NHL API
if not draft_picks_path.exists():
    print("Draft picks file does not exist. Pulling data from NHL API...")

    # for each year in the draft year range, fetch the draft picks data from the NHL API
    for year in range(draft_year_start, draft_year_end + 1):
        time.sleep(0.3)  # sleep for 0.3 seconds to avoid overwhelming the API

        # print debug message
        print(f"Fetching draft picks for year {year}...")

        try:
            # make a GET request to the NHL API to retrieve draft picks data
            response = requests.get(f"https://api-web.nhle.com/v1/draft/picks/{year}/all")
            
            # check if the request was successful (status code 200)
            if response.status_code == 200:

                # data appended to data list
                parsed_response = response.json()

                # append the draft year to each pick and add it to the data list
                # data list is not a nested list, but a flat list of dictionaries, each representing a draft pick
                for pick in parsed_response['picks']:
                    pick['draftYear'] = year
                    draft_picks.append(pick)
                
            # if the request does not return status code 200, raise an exception with the status code
            else:
                raise Exception(f"Failed to fetch draft picks data. Status code: {response.status_code}")

        # if any exception occurs during the request, raise an exception with the error message
        except Exception as e:
            raise Exception(f"An error occurred while fetching draft picks data from year {year}: {e}")
    
    # save the data to a JSON file
    with open(draft_picks_path, "w") as f:
        json.dump(draft_picks, f)

# if the file path does exist, load the data from the JSON file
else:

    # load the data from the JSON file
    print("Draft picks file exists. Loading data from file...")
    with open(draft_picks_path, "r") as f:
        draft_picks = json.load(f)

Draft picks file exists. Loading data from file...


In [6]:
len(draft_picks)

9512

Grab the player bio data

In [ ]:
# fetch the skater bios data from the NHL API if the file does not exist
if not skater_bios_path.exists():
    print("Skater bios file does not exist. Pulling data from NHL API...")

    # pull the skater bios data from the NHL API for the specified season range
    try:

        # make a GET request to the NHL API to retrieve skater bios data
        # f string is used to format the URL with the season year range
        response = requests.get(f"https://api.nhle.com/stats/rest/en/skater/bios?isAggregate=true&limit=-1&cayenneExp=seasonId>={season_year_start} and seasonId<={season_year_end} and gameTypeId=2")

        if response.status_code != 200:
            raise Exception(f"Failed to fetch skater bios data. Status code: {response.status_code}")

        response_json = response.json()
        length = response_json['total']
        skater_bios = response_json['data']

        if len(skater_bios) != length:
            raise Exception(f"Expected {length} skater bios, but got {len(skater_bios)}")

        # save the data to a JSON file
        with open(skater_bios_path, "w") as f:
            json.dump(skater_bios, f)

    except Exception as e:
        raise Exception(f"An error occurred while fetching skater bios data: {e}")

else:
    # load the data from the JSON file
    print("Skater bios file exists. Loading data from file...")
    with open(skater_bios_path, "r") as f:
        skater_bios = json.load(f)

# check the length of the skater bios data
print(f"Length of skater bios: {len(skater_bios)}")

Skater bios file exists. Loading data from file...
Length of skater bios: 6035


In [17]:
# fetch the skater stats data from the NHL API if the file does not exist

# list to hold the skater stats data
skater_stats = []

if not skater_stats_path.exists():
    print("Skater stats file does not exist. Pulling data from NHL API...")

    # for each year in the season year range, fetch the skater stats data from the NHL API
    for year in range(season_stats_year_start, season_stats_year_end + 1):

        print(f"Fetching skater stats for season {year}-{year + 1}...")

        # formatted like 19791980 where 1979 is the start year and 1980 is the end year
        season_id = (year * 10000) + (year + 1)  

        # sleep for 0.3 seconds to avoid overwhelming the API
        time.sleep(0.3)  

        # pull the skater stats data from the NHL API for the specified season range
        try:

            # make a GET request to the NHL API to retrieve skater stats data
            # f string is used to format the URL with the season year range
            response = requests.get(f"https://api.nhle.com/stats/rest/en/skater/summary?isAggregate=false&limit=-1&cayenneExp=seasonId={season_id} and gameTypeId=2")

            # check if the request was successful (status code 200)
            if response.status_code != 200:
                raise Exception(f"Failed to fetch skater stats data. Status code: {response.status_code}")

            # parse the json
            skater_stats_year = response.json()

            print(f"Total number of skater stats for season {year}-{year + 1}: {skater_stats_year['total']}")
            # check that the lengths of total and data are the same
            if skater_stats_year['total'] != len(skater_stats_year['data']):
                raise Exception(f"Mismatch in total skater stats count for season {year}-{year + 1}. Expected {skater_stats_year['total']}, got {len(skater_stats_year['data'])}")

            # append the skater stats data to the skater_stats list
            for skater in skater_stats_year['data']:
                skater_stats.append(skater)
            



        except Exception as e:
            raise Exception(f"An error occurred while fetching skater stats data: {e}")

    # save the data to a JSON file
    with open(skater_stats_path, "w") as f:
        json.dump(skater_stats, f)

else:
    # load the data from the JSON file
    print("Skater stats file exists. Loading data from file...")
    with open(skater_stats_path, "r") as f:
        skater_stats = json.load(f)

Skater stats file exists. Loading data from file...


In [19]:
skater_stats_length = len(skater_stats)
print(f"Length of skater stats: {skater_stats_length}")

Length of skater stats: 36815
